# [B1] How far does exact and alias matching resolve STTR research-institution names?

**Status:** exploratory; non-citable  
**Research question:** [B1 — STTR research-institution partner types](../../docs/research-questions.md#b1-sttr-partner-types); the dependency named for identity-resolved promotion of the B1 collaboration atlas  
**Decision this informs:** whether exact-and-alias resolution against the pinned IPEDS and NSF FFRDC lists collapses the `RI Name` surface strings enough to justify a versioned research-institution identity profile, or whether a curated alias list is the next step.  
**Data as of:** pinned by the raw SBIR.gov export sidecar or by `SBIR_AWARDS_SHA256` and `SBIR_AWARDS_AS_OF` (see input-status cell)  
**Owner:** STTR exploratory workbench

Descriptive question only. Among distinct `RI Name` strings on `Program = STTR` rows, what share resolves to one reference institution by exact name, by a derived main-campus name, or by a declared IPEDS alias, and what share is ambiguous, unresolved, or a placeholder? Reported per distinct string and per award.


## Interpretation boundary

1. **A resolved id is a name match, not a verified relationship.** It says the string names a listed institution. It does not say the institution performed the work.
2. **Grain is not decided here.** A string naming a laboratory, campus, foundation, or department inside an institution stays unresolved unless the reference list carries that unit as its own row. An FFRDC resolves to its own node, never to its administering university.
3. **Unresolved is not wrong.** It means neither the IPEDS name, its main-campus form, nor a specific declared alias matched. Spelling variants such as `Univ. of` are expected to land here in this first cut.
4. **No fuzzy matching.** The match rate is a floor for what curation could reach, not an estimate of true coverage.


## Data contract

- **Population:** raw export rows whose `Program` value is `STTR` after whitespace trimming and case folding, across all phases. One canonical award per `sbir_award_grain_key`; the `RI Name` of the representative edition is the surface string.
- **Grain:** distinct `RI Name` surface string for the match-rate table; canonical award for the award-weighted shares.
- **Keys:** the comparison key is `ORGANIZATION_KEY_V1` with a leading `THE` removed, applied identically to surface names and reference names. IPEDS rows are keyed by `UNITID`; FFRDC rows by a slug of the NSF name.
- **Inputs:** the raw SBIR.gov CSV at `SBIR_AWARDS_PATH` or `data/raw/sbir/award_data.csv`, verified through the canonical sidecar when present; `data/reference/sttr_partner_type_seed_lists/ipeds_institutions/HD2024.csv` and `.../ffrdc_master/ffrdc_master.csv` as pinned in `specs/sttr-spinout-linkage/seed-list-provenance.md`.
- **Exclusions:** none from the population. Placeholder strings (`N/A`, `NA`, `None`, `null`, `nan`, `-`) are reported as `blank`, not dropped.
- **Missingness:** a missing export yields a refusal and empty tables. A `blank` rule is typed missingness, not evidence of no partner.
- **Seed:** `20261029`; no stochastic operation.
- **Outputs:** in-notebook tables only. No artifact is written. Non-citable.


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "sbir_etl").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the sbir-analytics checkout")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from sbir_etl.extractors.sbir_award_export import (  # noqa: E402
    SbirGovSourceError,
    load_verified_award_export,
    read_sbir_gov_csv,
)
from sbir_etl.extractors.sbir_public_awards import normalize_sbir_awards  # noqa: E402
from sbir_etl.utils.data.file_io import file_sha256  # noqa: E402
from scripts.research_institutions.ri_resolver import (  # noqa: E402
    FFRDC_PATH,
    IPEDS_PATH,
    build_default_resolver,
    match_rate_table,
    resolution_frame,
)

RANDOM_SEED = 20261029
TOP_N = 40
AWARDS_PATH = Path(os.getenv("SBIR_AWARDS_PATH") or REPO_ROOT / "data" / "raw" / "sbir" / "award_data.csv")
DECLARED_SHA256 = os.getenv("SBIR_AWARDS_SHA256", "").strip().lower()
DECLARED_AS_OF = os.getenv("SBIR_AWARDS_AS_OF", "").strip()
REPO_ROOT


In [ ]:
def load_pinned_export(path: Path) -> tuple[dict[str, object], pd.DataFrame]:
    status: dict[str, object] = {
        "input": "raw SBIR.gov award_data.csv",
        "path": str(path),
        "exists": path.is_file(),
        "status": "REFUSED",
        "reason": "",
        "as_of": None,
        "sha256": None,
        "row_count": None,
        "basis": None,
    }
    empty = pd.DataFrame()
    if not path.is_file():
        status["reason"] = "source not found; set SBIR_AWARDS_PATH"
        return status, empty
    sidecar = path.with_suffix(".meta.json")
    if sidecar.is_file():
        try:
            verified = load_verified_award_export(path, sidecar)
        except (OSError, SbirGovSourceError) as exc:
            status["reason"] = f"canonical sidecar verification failed: {exc}"
            return status, empty
        meta = verified.metadata
        as_of = meta.upstream_published_at or meta.retrieved_at
        status.update(
            {
                "status": "READY",
                "as_of": as_of.date().isoformat(),
                "sha256": meta.sha256,
                "row_count": meta.row_count,
                "basis": "verified canonical sidecar",
            }
        )
        return status, verified.frame
    if not DECLARED_SHA256 or not DECLARED_AS_OF:
        status["reason"] = "sidecar absent; set SBIR_AWARDS_SHA256 and SBIR_AWARDS_AS_OF"
        return status, empty
    observed = file_sha256(path)
    if observed != DECLARED_SHA256:
        status["reason"] = "observed SHA-256 does not match SBIR_AWARDS_SHA256"
        status["sha256"] = observed
        return status, empty
    try:
        frame = read_sbir_gov_csv(path)
    except (OSError, SbirGovSourceError) as exc:
        status["reason"] = f"canonical CSV validation failed: {exc}"
        return status, empty
    status.update(
        {
            "status": "READY",
            "as_of": DECLARED_AS_OF,
            "sha256": observed,
            "row_count": len(frame),
            "basis": "operator-declared hash and as-of",
        }
    )
    return status, frame


snapshot_status, raw_source = load_pinned_export(AWARDS_PATH)
reference_status = pd.DataFrame(
    [
        {"input": "IPEDS HD2024", "path": str(IPEDS_PATH.relative_to(REPO_ROOT)), "exists": IPEDS_PATH.is_file()},
        {"input": "NSF FFRDC master", "path": str(FFRDC_PATH.relative_to(REPO_ROOT)), "exists": FFRDC_PATH.is_file()},
    ]
)
display(pd.DataFrame([snapshot_status]))
reference_status


## Surface strings on canonical STTR awards

`normalize_sbir_awards` selects one representative edition per canonical award key. The `RI Name` of that edition is joined back by source row, so one award contributes one surface string and one unit of award weight.


In [ ]:
analysis_ready = snapshot_status["status"] == "READY" and reference_status["exists"].all()

if analysis_ready:
    program = raw_source["Program"].fillna("").astype(str).str.strip().str.upper()
    sttr = raw_source.loc[program.eq("STTR")].copy()
    sttr["source_row"] = sttr.index + 2  # raw frame index is positional
    canonical = normalize_sbir_awards(sttr)
    chosen = canonical.loc[:, ["award_key", "source_row"]]
    portfolio = sttr.loc[chosen["source_row"] - 2, ["RI Name"]].reset_index(drop=True)
    portfolio["award_key"] = chosen["award_key"].to_numpy()
    surface_weights = (
        portfolio["RI Name"].fillna("").map(lambda value: " ".join(str(value).split()))
        .value_counts()
        .to_dict()
    )
    population = pd.DataFrame(
        [
            {"measure": "raw STTR source editions", "value": len(sttr)},
            {"measure": "canonical STTR awards", "value": len(portfolio)},
            {"measure": "distinct RI surface strings (whitespace-collapsed)", "value": len(surface_weights)},
        ]
    )
else:
    surface_weights = {}
    population = pd.DataFrame([{"measure": "status", "value": f"REFUSED — {snapshot_status['reason']}"}])
population


## Resolution by rule

Rules fire in order: `blank`, `exact_name`, `ipeds_main_campus`, `alias`; a rule that matches more than one institution yields `ambiguous`; otherwise `unresolved`. Shares are reported per distinct string and per award. The award-weighted share is the one that matters for the atlas, because a few institutions carry many awards.


In [ ]:
resolver = build_default_resolver() if analysis_ready else None
resolved = (
    resolution_frame(surface_weights.keys(), resolver, surface_weights)
    if analysis_ready
    else resolution_frame([], build_default_resolver())
)
rule_table = match_rate_table(resolved)
positive_rules = ["exact_name", "ipeds_main_campus", "alias"]
if analysis_ready:
    positive = resolved["rule"].isin(positive_rules)
    collapse = pd.DataFrame(
        [
            {"measure": "distinct strings with a single resolved id", "value": int(positive.sum())},
            {"measure": "distinct reference ids they resolve to", "value": int(resolved.loc[positive, "reference_id"].nunique())},
            {"measure": "awards covered by a single resolved id", "value": int(resolved.loc[positive, "awards"].sum())},
            {"measure": "share of awards covered", "value": float(resolved.loc[positive, "awards"].sum() / resolved["awards"].sum())},
            {"measure": "resolved ids from FFRDC list", "value": int(resolved.loc[positive & resolved["source"].eq("ffrdc"), "reference_id"].nunique())},
        ]
    )
else:
    collapse = pd.DataFrame()
display(rule_table)
collapse


## What stays unresolved

The heaviest unresolved strings by award count are the curation queue. Read them for the pattern (abbreviation, campus suffix, department prefix, foreign institution) before deciding whether an alias list or a grain rule is the next step. Ambiguous strings show which reference-name collisions matter in practice.


In [ ]:
def top_by_awards(frame: pd.DataFrame, rule: str, n: int = TOP_N) -> pd.DataFrame:
    subset = frame.loc[frame["rule"].eq(rule), ["surface_name", "key", "awards", "candidate_count"]]
    return subset.sort_values(["awards", "surface_name"], ascending=[False, True]).head(n).reset_index(drop=True)


display(top_by_awards(resolved, "unresolved"))
top_by_awards(resolved, "ambiguous")


## Resolved institutions by award count

The top resolved institutions show whether the exact rule is landing on the expected universities and FFRDCs, and which rule carried each.


In [ ]:
resolved_top = (
    resolved.loc[resolved["rule"].isin(positive_rules)]
    .groupby(["reference_id", "reference_name", "source"], as_index=False)
    .agg(awards=("awards", "sum"), surface_strings=("surface_name", "size"), rules=("rule", lambda s: ", ".join(sorted(set(s)))))
    .sort_values(["awards", "reference_name"], ascending=[False, True])
    .head(TOP_N)
    .reset_index(drop=True)
)
resolved_top


## Findings and caveats

| Claim | Evidence/artifact | Caveat or alternative explanation |
|---|---|---|
| _Fill from the rule table after a run on the pinned export_ | `rule_table`, `collapse` | Exact-and-alias only; the award-weighted share is a floor, not coverage. |
| _Fill from the unresolved queue_ | `top_by_awards(resolved, "unresolved")` | Abbreviations and campus suffixes are expected here; they are the curation list, not evidence of a missing institution. |

Separate observations from interpretation. A resolved id is a name match. It does not establish that the institution performed the work, and it does not settle the laboratory, campus, or department grain question.

## Promotion checklist

- [ ] Question and decision are explicit.
- [ ] Data snapshot, population, grain, keys, and exclusions are recorded.
- [ ] Samples and stochastic methods use a deterministic seed.
- [ ] The destination tier and its contract are explicit (a versioned `CompanyNameProfile` or a new identity primitive, with the grain rule frozen first).
- [ ] Recurring calculations have one canonical implementation.
- [ ] Findings and methodology are linked from `docs/`.
- [ ] Outputs and execution counts are cleared before commit.

Until promotion is complete, this notebook and its claims remain exploratory and non-citable.
